# 주13: RDD 변환

**202315163 김소희**

사각형의 가로와 세로 길이로 RDD를 만든다. `map()`을 이어 붙인 Pipeline으로 면적을 계산하고, 면적이 30 이상인지 판정해서 출력한다.

## 1. SparkSession 생성

환경변수 PYSPARK_PYTHON, PYSPARK_DRIVER_PYTHON에 지금 실행 중인 Python의 경로(`sys.executable`)를 넣는다. 그다음 SparkSession을 만들어 `spark` 변수에 저장한다.

In [1]:
import os
import sys
import pyspark

os.environ['PYSPARK_PYTHON']=sys.executable
os.environ['PYSPARK_DRIVER_PYTHON']=sys.executable

myConf=pyspark.SparkConf()
spark = pyspark.sql.SparkSession.builder\
    .master("local")\
    .appName("myApp")\
    .config(conf=myConf)\
    .getOrCreate()

## 2. RDD 생성

문제에 주어진 가로와 세로 길이를 한 줄씩 그대로 Python list에 넣는다. 이 list를 `sparkContext.parallelize()` 함수에 넘겨 RDD를 만든다.

In [2]:
myList=[
    "1, 2",
    "3, 4",
    "5, 6",
    "7, 8",
    "9, 10"
]
myRdd = spark.sparkContext.parallelize(myList)

`collect()` 함수로 RDD에 들어 있는 값을 확인한다.

가로와 세로가 `'1, 2'`처럼 문자열 하나로 붙어 있다. 곱하려면 먼저 쉼표로 나누고 정수로 바꿔야 한다.

In [3]:
myRdd.collect()

['1, 2', '3, 4', '5, 6', '7, 8', '9, 10']

## 3. map을 이용한 데이터 변환 Pipeline

먼저 면적을 판정하는 함수 `judge()`를 만든다.
면적이 30 이상이면 "30이상", 30 미만이면 "30미만"으로 판정하고, 면적과 판정 결과를 함께 돌려준다.

lambda에는 코드를 한 줄만 쓸 수 있다. 판정은 if, else로 여러 줄을 써야 해서 함수를 따로 만들었다.

In [4]:
def judge(area):
    if area >= 30:
        return (area, "30이상")
    else:
        return (area, "30미만")

`map()` 세 개와 `collect()`를 이어 붙여 Pipeline을 만든다. 앞 단계의 결과가 다음 단계의 입력이 된다.

- `map(lambda x: x.split(','))`: 한 줄을 쉼표로 나눈다. `'1, 2'`는 `['1', ' 2']`가 된다.
- (1) `map(lambda x: int(x[0]) * int(x[1]))`: 가로 `x[0]`와 세로 `x[1]`를 정수로 바꾼 뒤 곱해서 면적을 구한다.
- (2) `map(judge)`: 면적마다 `judge()` 함수를 호출해서 판정한다. 결과는 (면적, 판정 결과)이다.
- `collect()`: 결과를 list로 가져온다. `map()`은 transformation 함수라서 바로 계산되지 않고, action 함수인 `collect()`를 호출할 때 계산된다.

In [5]:
result = myRdd\
    .map(lambda x: x.split(','))\
    .map(lambda x: int(x[0]) * int(x[1]))\
    .map(judge)\
    .collect()

## 4. 면적과 판정 결과 출력

(3) `result`에서 하나씩 꺼내 사각형마다 면적과 판정 결과를 출력한다.

In [6]:
for i in result:
    print ("면적: {}, 판정: {}".format(i[0], i[1]))

면적: 2, 판정: 30미만
면적: 12, 판정: 30미만
면적: 30, 판정: 30이상
면적: 56, 판정: 30이상
면적: 90, 판정: 30이상
